In [1]:
!pip -q install scanpy humanize anndata

In [2]:
!pip install git+https://github.com/gillislab/pyMN#egg=pymetaneighbor

  Cloning https://github.com/gillislab/pyMN to /tmp/pip-install-1k1w6u6v/pymetaneighbor_570753f3fa1942c9871de53871b55202
  Running command git clone --filter=blob:none --quiet https://github.com/gillislab/pyMN /tmp/pip-install-1k1w6u6v/pymetaneighbor_570753f3fa1942c9871de53871b55202
  Resolved https://github.com/gillislab/pyMN to commit 882b54512347be66000f7909271fd8e0cb7def39
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pymetaneighbor: filename=pyMetaNeighbor-0.1.0-py3-none-any.whl size=29125 sha256=1dde0eb3270b148c4c5beac9d4d1f6ffb4a69e37fb92ae35254b023e99e310f9
  Stored in directory: /tmp/pip-ephem-wheel-cache-o9ewzwpr/wheels/81/4f/2f/ceba31de0b80b66b66844e8aa5fa3604b0f9754c5331b4a80e
  Created wheel for upsetplot: filename=upsetplot-0.9.0-py3-none-any.whl size=24864 sha256=57ea8633678c1a93e33cb98e2dae36745b8faf8c7177319733d6600bcb076

In [6]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import scipy.sparse as sp
import time
import os
import gc
import sys
import re
import resource
import time
import datetime
import multiprocessing

In [1]:
A1_h5ad = "/sbgenomics/project-files/GEN_A1/GEN_A1_pass2.h5ad"
A2_h5ad = "/sbgenomics/project-files/GEN_A2/250917_GEN_A2/GEN_A2_pass3.h5ad"
A3_h5ad = "/sbgenomics/project-files/GEN_A3/GEN_A3_pass2.h5ad" #A3 needs cortex filtering

In [6]:
#annotation_mapping_file = pd.read_csv("/sbgenomics/project-files//mappings_for_combined_five_PFC.20260117_000139/subtype_mappings.csv") #before PVM-micro
annotation_mapping_file = pd.read_csv("/sbgenomics/project-files//mappings_for_combined_five_PFC.20260130_210942/subtype_mappings.csv")

In [7]:
annotation_mapping_file

,subtype,subclass_annotation_v1,dataset,dataset_pass,base_folder,annotation_file,pass_num
0,12_4_1,Adaptive_Immune,GEN_A1,GEN_A1_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
1,12_1_1,Adaptive_Immune,GEN_A1,GEN_A1_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
2,12_1_2,Adaptive_Immune,GEN_A1,GEN_A1_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
3,12_2_1,Adaptive_Immune,GEN_A1,GEN_A1_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
4,12_1_3,Adaptive_Immune,GEN_A1,GEN_A1_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
...,...,...,...,...,...,...,...
977,8_3_4,VLMC,GEN_A3,GEN_A3_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
978,8_3_6,VLMC,GEN_A3,GEN_A3_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
979,8_3_2,VLMC,GEN_A3,GEN_A3_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2
980,8_3_3,VLMC,GEN_A3,GEN_A3_pass2,/sbgenomics/project-files/python_Metaneighbor_...,/sbgenomics/project-files/annotations/GENA1-3_...,2


In [8]:
annotation_mapping_file = annotation_mapping_file[["subtype", "subclass_annotation_v1", "dataset"]] 

In [4]:
#target_h5ad = "/sbgenomics/project-files/GEN_A13/260212_GEN_A13/GEN_A13_pass2.h5ad" 
#target_h5ad = "/sbgenomics/project-files/GEN_A4/260210_GEN_A4/GEN_A4_PFC_pass2.h5ad" 
#target_h5ad = "/sbgenomics/project-files/GEN_A16/GEN_A16_pass2.h5ad" 
target_h5ad = "/sbgenomics/project-files/GEN_A5/GEN_A5_pass2.h5ad" 

In [ ]:
# target_h5ad is now passed as the first command-line argument
# Example: python script.py /path/to/file.h5ad
if len(sys.argv) < 2:
    raise SystemExit(f"Usage: {sys.argv[0]} <target_h5ad>")
target_h5ad = sys.argv[1]

In [5]:
internal_dataset_study_id = re.search(r'GEN_\w+(?=_pass)', target_h5ad).group()
print(internal_dataset_study_id) 

GEN_A4_PFC


In [23]:
result_folder = "/sbgenomics/output-files/"+ internal_dataset_study_id + "_vrs_GEN_A1-3.cortex.subclass." + str(round(time.time()))

In [24]:
os.mkdir(result_folder)
print(result_folder)

/sbgenomics/output-files/GEN_A13_GEN_A1-3.cortex.subclass.1771537630


In [25]:
start_time = time.time()

In [26]:
internal_dataset = sc.read_h5ad(target_h5ad, backed="r")  # Open in read mode without loading everything

In [27]:
internal_dataset.obs

,n_genes,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mito,...,mito_genes,mito_ribo,ribo_genes,apoptosis,class,subclass,subtype,doublet_score,pred_dbl,demux_type
barcodekey,,,,,,,,,,,,,,,,,,,,,
R5573393_PFC_D19-8333-S11_AAACCCAAGAGGGTCT,4723,4723,8.460411,14342.0,9.571017,22.402733,30.023707,38.530191,52.182401,674.0,...,1.460477,-0.040214,-0.294412,-0.036237,7,7_1,7_1_1,0.008632,False,singlet
R5573393_PFC_D19-8333-S11_AAACGAAAGCTGTTCA,2058,2058,7.629976,5098.0,8.536800,26.853668,35.249117,46.116124,63.828953,63.0,...,-0.008764,-0.201737,0.073556,0.046336,4,4_2,4_2_2,0.026081,False,singlet
R5573393_PFC_D19-8333-S11_AAACGAAGTACTCGCG,6853,6853,8.832588,34194.0,10.439835,22.003860,29.013862,37.097152,50.757443,1067.0,...,1.032773,-0.000630,-0.431144,0.051247,6,6_1,6_1_3,0.009853,False,singlet
R5573393_PFC_D19-8333-S11_AAACGAAGTGGGTTGA,1559,1559,7.352441,3089.0,8.035926,20.977663,30.009712,42.376174,63.936549,19.0,...,-0.505265,-0.051362,-0.272635,-0.124683,1,1_1,1_1_1,0.013255,False,singlet
R5573393_PFC_D19-8333-S11_AAACGAATCTTCTAAC,4017,4017,8.298540,14083.0,9.552795,18.731804,26.535539,36.277782,52.708940,4.0,...,-1.391681,0.084965,-0.323495,0.041800,1,1_3,1_3_1,0.008632,False,singlet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R3898113_PFC_D19-5034-S8_TTTGGTTTCTTACACT,4257,4257,8.356555,12983.0,9.471473,20.349688,27.736271,36.971424,52.168220,29.0,...,-1.231922,-0.108979,-0.254111,0.090106,4,4_3,4_3_3,0.007003,False,singlet
R3898113_PFC_D19-5034-S8_TTTGTTGAGTTATGGA,5474,5474,8.607948,17844.0,9.789479,20.051558,26.283345,34.347680,47.993723,107.0,...,-0.922077,-0.142162,-0.052613,-0.175942,2,2_2,2_2_1,0.003779,False,singlet
R3898113_PFC_D19-5034-S8_TTTGTTGCAATGACCT,5787,5787,8.663542,19847.0,9.895859,17.267093,24.033859,32.322265,46.556155,91.0,...,-1.003114,0.052236,0.010609,-0.011807,4,4_4,4_4_1,0.008870,False,singlet


In [28]:
## Create a new AnnData object with only the raw matrix
internal_dataset = sc.AnnData(internal_dataset.raw.X, obs=internal_dataset.obs, var=internal_dataset.var)
internal_dataset = internal_dataset.to_memory()

In [29]:
internal_dataset

AnnData object with n_obs × n_vars = 1459327 × 27370
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'Batch', 'individualID', 'libraryID', 'Source', 'brain_region', 'subject', 'dataset', 'new_batch', 'Channel', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type'
    var: 'gene_symbols', 'feature_types', 'gene_id', 'gene_name', 'mito', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'protein_coding', 'mitocarta', 'robust_protei

In [30]:
internal_dataset.var = internal_dataset.var.set_index("gene_id")

In [1]:
# Create cell_type column
internal_dataset.obs['cell_type'] = internal_dataset.obs['subtype'] #or class
internal_dataset.obs['level_1'] = internal_dataset.obs['class'] #or class
internal_dataset.obs['level_2'] = internal_dataset.obs['subclass']
internal_dataset.obs['level_3'] = internal_dataset.obs['subtype'] #or class
internal_dataset.obs['study_id'] = internal_dataset_study_id
#TODO fix this setup
internal_dataset.obs['subclass_annotation_v1'] = internal_dataset.obs['subtype'] #or class

NameError: name 'internal_dataset' is not defined

In [15]:
def load_h5ad_sample(h5ad_path, study_id, sample_fraction=1.0, random_seed=42):
    """
    Load h5ad file with optional random sampling of cells.
    
    Parameters:
    -----------
    h5ad_path : str
        Path to the h5ad file
    study_id : str
        Study ID to assign to obs['study_id']
    sample_fraction : float, default=1.0
        Fraction of cells to load (0.0 to 1.0). Use 1.0 to load all cells.
    random_seed : int, default=42
        Random seed for reproducibility. Set to None for no seed.
    
    Returns:
    --------
    AnnData object with cells loaded into memory
    """
    # Open in read mode without loading everything
    adata = sc.read_h5ad(h5ad_path, backed="r")
    
    # Get total number of cells
    n_cells = adata.shape[0]
    
    if sample_fraction < 1.0:
        # Randomly sample cells
        if random_seed is not None:
            np.random.seed(random_seed)
        sample_size = int(n_cells * sample_fraction)
        random_indices = np.random.choice(n_cells, size=sample_size, replace=False)
        random_indices = np.sort(random_indices)
        
        # Load only the sampled cells
        adata = sc.AnnData(adata.raw.X[random_indices, :], 
                          obs=adata.obs.iloc[random_indices], 
                          var=adata.var)
    else:
        # Load all cells
        adata = sc.AnnData(adata.raw.X, obs=adata.obs, var=adata.var)
    
    adata = adata.to_memory()
    adata.var = adata.var.set_index("gene_id")
    adata.obs['study_id'] = study_id
    
    return adata

In [ ]:
GEN_A1_sample_fraction = 0.5
A1 = load_h5ad_sample(A1_h5ad, "GEN_A1", sample_fraction=GEN_A1_sample_fraction)

In [ ]:
#A2 = load_h5ad_sample(A2_h5ad, "GEN_A2", sample_fraction=0.05)
A2 = load_h5ad_sample(A2_h5ad, "GEN_A2", sample_fraction=1.0)

In [ ]:
A1

In [ ]:
A2

In [ ]:
A3 = sc.read_h5ad(A3_h5ad, backed="r")  # Open in read mode without loading everything

In [ ]:
# Find indices where brain_region is PFC, PMC, or PVC
region_mask = A3.obs['brain_region'].isin(['PFC', 'PMC', 'PVC'])
region_indices = np.where(region_mask)[0]
print(f"Found {len(region_indices)} cells with PFC/PMC/PVC in brain_region out of {A3.n_obs} total")

In [ ]:
# Extract the subset
X_subset = A3.X[region_indices, :]
var_subset = A3.var
A3 = sc.AnnData(
    X_subset,
    obs=A3.obs.iloc[region_indices],
    var=var_subset
)
A3 = A3.to_memory()
A3.var = A3.var.set_index("gene_id")
print(f"Loaded {A3.n_obs} PFC/PMC/PVC cells into memory")

In [ ]:
A3.obs['study_id'] = "GEN_A3_ctx"

In [7]:
h5ad_inputs = pd.DataFrame({
    "h5ad_var": ["target_h5ad", "A1_h5ad", "A2_h5ad", "A3_h5ad"],
    "dataset_name": [internal_dataset_study_id, "GEN_A1", "GEN_A2", "GEN_A3_ctx"],
    "sample_fraction": [1.0, GEN_A1_sample_fraction, 1.0, 1.0],
    "h5ad_path": [target_h5ad, A1_h5ad, A2_h5ad, A3_h5ad],
})

h5ad_inputs.to_csv(os.path.join(result_folder, "h5ad_inputs.csv"), index=False)
print(h5ad_inputs)

NameError: name 'internal_dataset_study_id' is not defined

In [ ]:
def add_subclass_annotation_inplace(
    adata,
    mapping: pd.DataFrame,
    *,
    subtype_col: str = "subtype",
    subclass_col: str = "subclass_annotation_v1",
    dataset_col: str = "dataset",
    dataset_key,  # filters mapping[dataset_col] == dataset_key
    out_col: str = "subclass_annotation_v1",
    drop_unmapped: bool = True,
):
    """
    Memory-friendly mapping of adata.obs[subtype_col] -> subclass annotation.

    - Optionally subsets the mapping by dataset_key.
    - Adds one column (out_col) to adata.obs using Series.map (no merge).
    - Prints how many cells did not receive a mapping.
    - Optionally drops unmapped cells in-place to avoid copying large matrices.

    Returns the same AnnData object (modified in place).
    """
    # Keep only needed columns and drop duplicates to minimize memory
    cols = [subtype_col, subclass_col]
    if dataset_col in mapping.columns:
        cols.append(dataset_col)

    m = mapping.loc[:, cols].drop_duplicates()

    if dataset_key is not None:
        if dataset_col not in m.columns:
            raise ValueError(f"`dataset_key` provided but mapping has no column {dataset_col!r}.")
        m = m[m[dataset_col].astype(str).eq(str(dataset_key))]

    # Build mapper: subtype -> subclass
    mapper = (
        m.dropna(subset=[subtype_col, subclass_col])
         .set_index(subtype_col)[subclass_col]
    )

    # Map without merge (low memory)
    adata.obs[out_col] = adata.obs[subtype_col].map(mapper)

    n_total = adata.n_obs
    n_unmapped = int(adata.obs[out_col].isna().sum())
    n_mapped = n_total - n_unmapped
    print(f"[add_subclass_annotation_inplace] Mapped: {n_mapped:,} / {n_total:,} cells; "
          f"Unmapped: {n_unmapped:,} ({(n_unmapped / max(1, n_total)):.1%})")

    if drop_unmapped and n_unmapped > 0:
        mask = adata.obs[out_col].notna().to_numpy()
        adata._inplace_subset_obs(mask)

    return adata

In [ ]:
add_subclass_annotation_inplace(A1, annotation_mapping_file, dataset_key="GEN_A1")

In [ ]:
add_subclass_annotation_inplace(A2, annotation_mapping_file, dataset_key="GEN_A2")

In [ ]:
add_subclass_annotation_inplace(A3, annotation_mapping_file, dataset_key="GEN_A3")

In [ ]:
merged=ad.concat([internal_dataset, A1, A2, A3], join="inner")

In [ ]:
merged.obs['cell_type'] = merged.obs['subclass_annotation_v1'].to_numpy(dtype="str")

In [ ]:
merged.obs.index = merged.obs.index.to_numpy(dtype="str")

In [ ]:
pymn.variableGenes(merged, study_col='study_id')

In [ ]:
sum(merged.var.highly_variable)

In [ ]:
merged = merged[:, merged.var.highly_variable]

In [ ]:
#use numpy types instead of pandas
merged.var.highly_variable = merged.var.highly_variable.to_numpy(dtype="bool")
merged.var.index = merged.var.index.to_numpy(dtype="str")

In [ ]:
#make cell indices unique
merged.obs_names_make_unique()

In [ ]:
merged.obs['cell_type'] = merged.obs['cell_type'].to_numpy(dtype="str")
merged.obs['study_id'] = merged.obs['study_id'].to_numpy(dtype="str")
merged.obs.index = merged.obs.index.to_numpy(dtype="str")

In [ ]:
print("Before running metaneighbor all vs all")

In [ ]:
#%%time
#run
pymn.MetaNeighborUS(merged,
                    study_col='study_id',
                    ct_col='cell_type',
                    fast_version=True, symmetric_output=True)

In [ ]:
print("After running metaneighbor all vs all")
aurocs = merged.uns["MetaNeighborUS"]
aurocs.to_csv(result_folder + "/aurocs_full.csv.gz", compression="gzip")

In [ ]:
#seems to crash here, not sure why - moved to python script
#run 1 vs best
pymn.MetaNeighborUS(merged,
                    study_col='study_id',
                    ct_col='cell_type', one_vs_best=True,
                    fast_version=True, symmetric_output=True)

In [ ]:
aurocs = merged.uns["MetaNeighborUS_1v1"]
aurocs.to_csv(result_folder + "/aurocs_1v1.csv.gz", compression="gzip")

In [ ]:
cell_counts = merged.obs.groupby("study_id").size()
cell_counts.to_csv(result_folder + "/cell_study_counts.csv")
cell_type_counts = merged.obs[["study_id", "cell_type"]].drop_duplicates().groupby("study_id").size()
cell_type_counts.to_csv(result_folder + "/cell_type_per_study_counts.csv")

In [ ]:
for set_threshold in [0.8, 0.9, 0.95, 0.99, 0.999]:
    print(set_threshold)
    pymn.topHits(merged, threshold=set_threshold)
    tophit_table = merged.uns['MetaNeighborUS_topHits']
    tophit_table.to_csv(result_folder + "/top_hits."+str(set_threshold)+".csv")

In [ ]:
merged.obs.to_csv(result_folder + "/merged.obs.csv.gz", compression="gzip")
merged.var.to_csv(result_folder + "/merged.var.csv.gz", compression="gzip")

In [ ]:
#write number of unique genes per cell
if sp.issparse(merged.X):
    n_unique_genes = np.asarray((merged.X > 0).sum(axis=1)).ravel()
else:
    n_unique_genes = (merged.X > 0).sum(axis=1).astype(int)
merged.obs["n_unique_genes"] = n_unique_genes
merged.obs["celltype_study"] = (
    merged.obs["study_id"].astype(str) + "|" + merged.obs["cell_type"].astype(str)
)
avg_genes_per_group = (
    merged.obs.groupby("celltype_study")["n_unique_genes"].mean().sort_index()
)
print(avg_genes_per_group.head())
avg_genes_per_group.reset_index().to_csv(result_folder + "/avg_unique_genes_per_celltype_study.csv", index=False)

In [ ]:
#write out peak memory at end
mem_usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# print the memory usage in megabytes
print("Peak memory use in Gb  " + str(round(mem_usage / 1024 / 1024,2 )) + " PID  " + str(os.getpid()))

os.mkdir(os.path.join(result_folder, "Peak memory use in Gb " + str(round(mem_usage / 1024 / 1024 ,2))))

In [ ]:
end_time = time.time()
print("Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0])
os.mkdir(os.path.join(result_folder, "Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0]))

In [ ]:
print("Done!")

In [ ]:
print(result_folder)